# Решение тестового задания от Титова Александра

# Загрузка библиотек, фиксирование random seed

In [50]:
!chcp 65001
!python --version

Active code page: 65001
Python 3.13.11


In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import time
import gc

import pickle
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    Doc
)

import nltk
from nltk.corpus import stopwords

In [3]:
SEED = 0
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# версии библиотек

In [53]:
print("Версии используемых библиотек:")
import bs4
print(f"bs4: {bs4.__version__}")
import sentence_transformers
print(f"sentence_transformers: {sentence_transformers.__version__}")
import sklearn
print(f"sklearn: {sklearn.__version__}")
import pyarrow
print(f"pyarrow: {pyarrow.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"torch: {torch.__version__}")
print(f"nltk: {nltk.__version__}")

print("rank_bm25: 0.2.2")
print("natasha: 1.6.0")

Версии используемых библиотек:
bs4: 4.15.0
sentence_transformers: 5.6.0
sklearn: 1.8.0
pyarrow: 23.0.0
pandas: 2.3.3
numpy: 2.3.5
torch: 2.9.1+cu130
nltk: 3.10.0
rank_bm25: 0.2.2
natasha: 1.6.0


In [54]:
!pip show rank-bm25 | findstr Version

Version: 0.2.2


In [55]:
!pip show natasha | findstr Version

Version: 1.6.0


# Загрузка данных, просмотр первых строк и структуры данных

In [8]:
articles = pd.read_feather("data/candidate_data/articles.f")
print("articles.f загружен, размер:", articles.shape)
calibration = pd.read_feather("data/candidate_data/calibration.f")
print("calibration.f загружен, размер:", calibration.shape)
test = pd.read_feather("data/candidate_data/test.f")
print("test.f загружен, размер:", test.shape)

articles.f загружен, размер: (793, 3)
calibration.f загружен, размер: (500, 3)
test.f загружен, размер: (500, 2)


In [9]:
if articles is not None:
    print("--- articles.f ---")
    display(articles.head())
    display(articles.info())

--- articles.f ---


,article_id,title,body
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://w..."
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при вх..."
2,1747,Не допустить блокировки профиля,<ol><li><p><strong>Не заводите несколько аккау...
3,1774,Оставить или удалить профиль,<p>⚡ Не удаляйте профиль с подтверждёнными дан...
4,1775,Удалить профиль,"<p>⚡ Удалить профиль не получится, если у вас ..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   article_id  793 non-null    int64 
 1   title       793 non-null    string
 2   body        793 non-null    string
dtypes: int64(1), string(2)
memory usage: 18.7 KB


None

In [10]:
if calibration is not None:
    print("--- calibration.f ---")
    display(calibration.head())
    display(calibration.info())

--- calibration.f ---


,query_id,query_text,ground_truth
0,1,Как передать товар через службу авито,1909 4234
1,2,"Можете подсказать, если заказать товар Авито д...",2865 4400
2,3,Здравствуйте. Как отправить товар через Авито.,1909
3,4,как получить деньги за возрат если продавец уж...,4400 4403
4,5,"Когда мне прийдут деньги за доставку, сегодня ...",4361


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   query_id      500 non-null    int64 
 1   query_text    500 non-null    string
 2   ground_truth  500 non-null    string
dtypes: int64(1), string(2)
memory usage: 11.8 KB


None

In [11]:
if test is not None:
    print("--- test.f ---")
    display(test.head())
    display(test.info())

--- test.f ---


,query_id,query_text
0,1,"Здравствуйте! Подскажите, пожалуйста, не могу ..."
1,2,"Здравствуйте , почему так долго доставляется в..."
2,3,"Здравствуйте,подскажите как мне отправить крос..."
3,4,Здравствуйте! В каких случаях за возврат снима...
4,5,Почему у меня доставки в несколько раз дороже ...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   query_id    500 non-null    int64 
 1   query_text  500 non-null    string
dtypes: int64(1), string(1)
memory usage: 7.9 KB


None

# Проверка наличия колонок

In [12]:
if articles is not None:
    required_articles_cols = ['article_id', 'title', 'body']
    missing = [col for col in required_articles_cols if col not in articles.columns]
    if missing:
        print(f"В articles.f отсутствуют колонки: {missing}")
    else:
        print("Все необходимые колонки в articles.f присутствуют.")

Все необходимые колонки в articles.f присутствуют.


In [13]:
if calibration is not None:
    required_calib_cols = ['query_id', 'query_text', 'ground_truth']
    missing = [col for col in required_calib_cols if col not in calibration.columns]
    if missing:
        print(f"В calibration.f отсутствуют колонки: {missing}")
    else:
        print("Все необходимые колонки в calibration.f присутствуют.")

Все необходимые колонки в calibration.f присутствуют.


In [14]:
if test is not None:
    required_test_cols = ['query_id', 'query_text']
    missing = [col for col in required_test_cols if col not in test.columns]
    if missing:
        print(f"В test.f отсутствуют колонки: {missing}")
    else:
        print("Все необходимые колонки в test.f присутствуют.")

Все необходимые колонки в test.f присутствуют.


# очистка текста

In [15]:
def clean_html(html_text):
    if not isinstance(html_text, str):
        return ""

    soup = BeautifulSoup(html_text, 'html.parser')

    for script in soup(["script", "style"]):
        script.decompose()

    text = soup.get_text(separator=' ', strip=True)

    text = ' '.join(text.split())
    return text

# Функция формирования полного текста статьи (заголовок + тело)
def build_full_text(row, duplicate_title=False):
    title = row['title'] if isinstance(row['title'], str) else ""
    body_cleaned = clean_html(row['body']) if isinstance(row.get('body'), str) else ""
    if duplicate_title:
        full = f"{title} {title} {body_cleaned}".strip()
    else:
        full = f"{title} {body_cleaned}".strip()
    return full

In [16]:
# тут применяется к articles с дублированием заголовка для улучшения поиска
# а duplicate_title=True для усиления веса заголовка
articles['full_text'] = articles.apply(lambda row: build_full_text(row, duplicate_title=True), axis=1)

print(f"Создана колонка 'full_text' для {len(articles)} статей.")
print("Примеры полных текстов (первые 3):")
for i in range(min(3, len(articles))):
    print(f"Статья {i+1}: {articles['full_text'].iloc[i][:200]}...\n")

# Проверка длины текстов
articles['text_len'] = articles['full_text'].str.len()
print(f"Статистика длины текстов (символов):")
print(articles['text_len'].describe())

articles.drop(columns=['text_len'], inplace=True)

Создана колонка 'full_text' для 793 статей.
Примеры полных текстов (первые 3):
Статья 1: Имя или название компании Имя или название компании Зайдите в раздел Управление профилем . Нажмите на карандашик в правом верхнем углу. Введите новое имя или название и нажмите Сохранить . ⚡ Важно: на...

Статья 2: Понять, что профиль заблокирован Понять, что профиль заблокирован Проверьте, какое сообщение вы видите при входе в профиль. Профиль заблокирован Доступ к профилю ограничен Доступ ограничен. Пройдите п...

Статья 3: Не допустить блокировки профиля Не допустить блокировки профиля Не заводите несколько аккаунтов для продаж в одной категории. Так поступают пользователи, которые хотят обойти наши правила, — то есть п...

Статистика длины текстов (символов):
count       793.000000
mean       5195.970996
std       19118.332684
min          74.000000
25%        1209.000000
50%        2307.000000
75%        4836.000000
max      506180.000000
Name: text_len, dtype: float64


По статистике видно:

Разброс очень большой: от 74 символов до 506 тысяч. Т. е. есть короткие подсказки и есть объёмные инструкции с таблицами или списками.

Среднее (5196) сильно выше медианы (2307), что говорит о наличии нескольких очень длинных статей, которые меняют среднее.

Стандартное отклонение (19118) подтверждает высокую вариативность.

Минимум 74 символа это, скорее всего, статья с одним предложением, но не пустая.

# Настройки путей и флага для загрузки данных токенизированного корпуса и индекса из весов

In [17]:
USE_CACHE = True  # True — загрузить из файлов, False — пересобрать заново
CACHE_DIR = "assets"
CORPUS_PATH = os.path.join(CACHE_DIR, "tokenized_corpus.pkl")
BM25_PATH = os.path.join(CACHE_DIR, "bm25_index.pkl")

In [18]:
os.makedirs(CACHE_DIR, exist_ok=True)

# Токенизация и лемматизация для BM25, загрузка или построение и сохранение

In [29]:
def tokenize(
    text, 
    segmenter=Segmenter(),
    morph_tagger=NewsMorphTagger(NewsEmbedding()),
    morph_vocab=MorphVocab(),
    stop_words=set(stopwords.words('russian'))
):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    
    tokens = []
    for token in doc.tokens:
        if token.text.isalpha():
            token.lemmatize(morph_vocab)
            lemma = token.lemma.lower()

            if lemma not in stop_words and len(lemma) >= 2:
                tokens.append(lemma)
                
    return tokens

In [32]:
USE_CACHE = True

In [33]:
if USE_CACHE and os.path.exists(CORPUS_PATH):
    print(f"Загрузка токенизированного корпуса из {CORPUS_PATH}...")
    with open(CORPUS_PATH, "rb") as f:
        tokenized_corpus = pickle.load(f)
    print(f"Корпус загружен. Всего статей: {len(tokenized_corpus)}")
else:
    print("Флаг кэша отключен или файлы не найдены. Начинаем обработку с нуля...")

    stop_words = set(stopwords.words('russian'))

    print(f"Загружено {len(stop_words)} стоп-слов.")
    print("Начинаем токенизацию статей...")
    
    tokenized_corpus = []
    for idx, text in enumerate(articles['full_text']):
        # Просто вызываем внешнюю функцию
        tokens = tokenize(text)
        tokenized_corpus.append(tokens)
        
        if (idx + 1) % 100 == 0:
            print(f"Обработано {idx + 1} статей из {len(articles)}")

    print(f"Токенизация завершена. Всего статей: {len(tokenized_corpus)}")

    with open(CORPUS_PATH, "wb") as f:
        pickle.dump(tokenized_corpus, f)
    print(f"Токенизированный корпус сохранён в {CORPUS_PATH}")

Загрузка токенизированного корпуса из assets\tokenized_corpus.pkl...
Корпус загружен. Всего статей: 793


In [20]:
print("Пример токенов для первой статьи (первые 20):")
print(tokenized_corpus[0][:20])

token_counts = [len(tokens) for tokens in tokenized_corpus]
print()
print("Статистика количества токенов в статьях:")
print(pd.Series(token_counts).describe())

Пример токенов для первой статьи (первые 20):
['имя', 'название', 'компания', 'имя', 'название', 'компания', 'зайдите', 'раздел', 'управление', 'профиль', 'нажать', 'карандашик', 'правый', 'верхний', 'угол', 'введите', 'новый', 'имя', 'название', 'нажать']

Статистика количества токенов в статьях:
count      793.000000
mean       493.016393
std       2264.599100
min          3.000000
25%        113.000000
50%        210.000000
75%        442.000000
max      61667.000000
dtype: float64


# BM25-индекс загрузка или построение и сохранение

In [21]:
if USE_CACHE and os.path.exists(BM25_PATH):
    print(f"Загрузка BM25-индекса из {BM25_PATH}...")
    
    with open(BM25_PATH, "rb") as f:
        bm25 = pickle.load(f)
    
    print("BM25-индекс успешно восстановлен из кэша.")

else:
    print("Флаг кэша отключен или файл индекса не найден. Строим BM25-индекс с нуля...")
    
    if tokenized_corpus is None:
        raise ValueError("Ошибка: tokenized_corpus не найден. Сначала загрузите или создайте корпус!")
    
    bm25 = BM25Okapi(tokenized_corpus)
    print("BM25-индекс построен.")
    
    with open(BM25_PATH, "wb") as f:
        pickle.dump(bm25, f)
        
    print(f"BM25-индекс сохранён в {BM25_PATH}")

Загрузка BM25-индекса из assets\bm25_index.pkl...
BM25-индекс успешно восстановлен из кэша.


In [22]:
print(f"Количество документов: {bm25.corpus_size}")
print(f"Средняя длина документа: {bm25.avgdl:.2f}")

Количество документов: 793
Средняя длина документа: 493.02


# Генерация эмбеддингов статей

In [23]:
MODEL_NAME = "deepvk/USER2-small" 
BATCH_SIZE = 32
CHUNK_SIZE = 1000
EMBED_PATH = "assets/doc_embeddings.npy"

In [24]:
os.makedirs(os.path.dirname(EMBED_PATH), exist_ok=True)

In [25]:
print(articles.columns.tolist())


['article_id', 'title', 'body', 'full_text']


In [ ]:
# жесткая экономии видеопамяти
MODEL_NAME = "deepvk/USER2-small" 
BATCH_SIZE = 4          # Маленький батч для GPU, чтобы длинные тексты не забивали память (а то было OOM на 6 ГБ видеокарте)
CHUNK_SIZE = 100        # Обработка толко 100 статей за раз
USE_CACHE = True
EMBED_PATH = "assets/doc_embeddings.npy"

# конфигурация аллокатора PyTorch задается ДО загрузки каких-либо тензоров
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

os.makedirs(os.path.dirname(EMBED_PATH), exist_ok=True)
doc_embeddings = None

if USE_CACHE and os.path.exists(EMBED_PATH):
    print(f"Найдены сохранённые эмбеддинги в {EMBED_PATH}. Загружаем...")
    doc_embeddings = np.load(EMBED_PATH)
    print(f"Эмбеддинги успешно загружены. Форма матрицы: {doc_embeddings.shape}")

else:
    print(f"Флаг кэша отключен или файл не найден. Загрузка модели {MODEL_NAME} на GPU...")
    
    # очищистка GPU перед стартом
    gc.collect()
    torch.cuda.empty_cache()
    
    start_time = time.time()
    model = SentenceTransformer(MODEL_NAME, device="cuda")
    print(f"Модель загружена за {time.time() - start_time:.2f} сек.")
    
    # урезаю тут максимальную длину контекста с 8192 до 1024
    # чтоб длинные статьи не вызывали ошибку Out of Memory
    model.max_seq_length = 1024 
    print(f"Контекст модели принудительно ограничен до: {model.max_seq_length}")
    
    print("Подготовка текстов статей...")
    raw_texts = articles['full_text'].tolist()
    texts = [
        str(text) if isinstance(text, str) and len(text.strip()) > 0 else " " 
        for text in raw_texts
    ]
    
    total_texts = len(texts)
    print(f"Начинаем кодирование {total_texts} текстов частями по {CHUNK_SIZE}...")
    
    all_chunks = []
    start_time = time.time()

    # Включаю контекстный менеджер автокаста в FP16 для двукратной экономии памяти GPU
    with torch.cuda.amp.autocast():
        for i in range(0, total_texts, CHUNK_SIZE):
            chunk_texts = texts[i : i + CHUNK_SIZE]
            print(f"Обработка статей с {i} по {min(i + CHUNK_SIZE, total_texts)} из {total_texts}...")
            
            # Кодируем текущий чанк
            chunk_embeddings = model.encode(
                chunk_texts,
                batch_size=BATCH_SIZE,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=True
            )
            all_chunks.append(chunk_embeddings)
            
            # Очистка памяти после каждого чанка
            del chunk_embeddings
            gc.collect()
            torch.cuda.empty_cache()

    doc_embeddings = np.vstack(all_chunks)

    print(f"Кодирование успешно завершено за {time.time() - start_time:.2f} сек.")
    print(f"Итоговая форма эмбеддингов: {doc_embeddings.shape}")

    np.save(EMBED_PATH, doc_embeddings)
    print(f"Эмбеддинги сохранены в {EMBED_PATH}")


Флаг кэша отключен или файл не найден. Загрузка модели deepvk/USER2-small на GPU...


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Модель загружена за 7.79 сек.
Контекст модели принудительно ограничен до: 1024
Подготовка текстов статей...
Начинаем кодирование 793 текстов частями по 100...
Обработка статей с 0 по 100 из 793...
Обработка статей с 100 по 200 из 793...
Обработка статей с 200 по 300 из 793...
Обработка статей с 300 по 400 из 793...
Обработка статей с 400 по 500 из 793...
Обработка статей с 500 по 600 из 793...
Обработка статей с 600 по 700 из 793...
Обработка статей с 700 по 793 из 793...
Кодирование успешно завершено за 9.02 сек.
Итоговая форма эмбеддингов: (793, 384)
Эмбеддинги сохранены в assets/doc_embeddings.npy


# Функция поиска для одного запроса

In [34]:
EPS = 1e-8

def retrieve(query_text, alpha=0.5):
    query_tokens = tokenize(query_text)
    if not query_tokens:
        return ""
    
    bm25_scores = bm25.get_scores(query_tokens)
    
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    if bm25_max - bm25_min < EPS:
        bm25_norm = np.zeros_like(bm25_scores)
    else:
        bm25_norm = (bm25_scores - bm25_min) / (bm25_max - bm25_min + EPS)
    
    query_emb = model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)[0]
    
    dense_scores = cosine_similarity([query_emb], doc_embeddings)[0]  # shape (n_docs,)
    
    dense_min, dense_max = dense_scores.min(), dense_scores.max()
    if dense_max - dense_min < EPS:
        dense_norm = np.zeros_like(dense_scores)
    else:
        dense_norm = (dense_scores - dense_min) / (dense_max - dense_min + EPS)
    
    final_scores = alpha * bm25_norm + (1 - alpha) * dense_norm
    
    top_indices = np.argsort(final_scores)[::-1][:10]
    
    top_article_ids = articles['article_id'].iloc[top_indices].tolist()
    
    return ' '.join(map(str, top_article_ids))

In [35]:
if calibration is not None:
    first_query = calibration['query_text'].iloc[0]
    print(f"Пример запроса: {first_query}")
    result = retrieve(first_query, alpha=0.5)
    print(f"Результат: {result}")
    print(f"Количество найденных статей: {len(result.split())}")

Пример запроса: Как передать товар через службу авито
Результат: 4234 1909 1951 4009 1923 1954 4387 4286 2408 4407
Количество найденных статей: 10


# Функция расчёта MAP@10

In [38]:
def ap_at_10(ground_truth_list, predicted_list):
    if not ground_truth_list:
        return 0.0
    
    ground_truth_set = set(ground_truth_list)
    predicted = predicted_list[:10]
    
    hits = 0
    sum_precision = 0.0
    for i, p in enumerate(predicted, start=1):
        if p in ground_truth_set:
            hits += 1
            precision_at_i = hits / i
            sum_precision += precision_at_i
    
    k = min(len(ground_truth_list), 10)
    if k == 0:
        return 0.0
    return sum_precision / k

In [39]:
# проверка
test_gt = [1909, 4234]
test_pred = [4234, 1909, 1951, 4009, 1923]
ap = ap_at_10(test_gt, test_pred)
print(f"AP@10 для примера: {ap:.4f}")  # должно быть 1.0

AP@10 для примера: 1.0000


# Подбор гиперпараметра alpha на калибровочных данных

In [ ]:
if calibration is not None:
    #Преобразует ground_truth из строк в списки int
    calibration['ground_truth_list'] = calibration['ground_truth'].apply(
        lambda x: [int(i) for i in str(x).split()] if pd.notna(x) else []
    )
    
    # Сетка значений alpha для перебора
    alpha_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    
    print("Начинаем подбор alpha на калибровочных данных...")
    results = []
    
    for alpha in alpha_values:
        total_ap = 0.0
        count = 0

        # вычисляет AP@10 для каждого запроса 
        for idx, row in calibration.iterrows():
            query = row['query_text']
            gt = row['ground_truth_list']
            if not gt:  # если нет правильных ответов, пропуск
                continue
            pred_str = retrieve(query, alpha=alpha)
            pred_list = [int(x) for x in pred_str.split()] if pred_str else []
            ap = ap_at_10(gt, pred_list)
            total_ap += ap
            count += 1
        map_score = total_ap / count if count > 0 else 0.0
        results.append((alpha, map_score))
        print(f"alpha={alpha:.1f} -> MAP@10 = {map_score:.4f}")
    
    # лучший alpha
    best_alpha, best_map = max(results, key=lambda x: x[1])
    print(f"\nЛучший alpha: {best_alpha:.1f} с MAP@10 = {best_map:.4f}")
    
    results_df = pd.DataFrame(results, columns=['alpha', 'MAP@10'])
    print("\nТаблица результатов:")
    print(results_df)
else:
    best_alpha = 0.5 
    print("calibration не загружен, используется alpha=0.5 по умолчанию.")

print(f"Итоговое значение alpha для теста: {best_alpha}")

Начинаем подбор alpha на калибровочных данных...
alpha=0.0 -> MAP@10 = 0.2399
alpha=0.1 -> MAP@10 = 0.3122
alpha=0.2 -> MAP@10 = 0.3715
alpha=0.3 -> MAP@10 = 0.3925
alpha=0.4 -> MAP@10 = 0.3993
alpha=0.5 -> MAP@10 = 0.3800
alpha=0.6 -> MAP@10 = 0.3582
alpha=0.7 -> MAP@10 = 0.3411
alpha=0.8 -> MAP@10 = 0.3214
alpha=0.9 -> MAP@10 = 0.3003
alpha=1.0 -> MAP@10 = 0.2848

Лучший alpha: 0.4 с MAP@10 = 0.3993

Таблица результатов:
    alpha    MAP@10
0     0.0  0.239857
1     0.1  0.312239
2     0.2  0.371494
3     0.3  0.392544
4     0.4  0.399335
5     0.5  0.379972
6     0.6  0.358158
7     0.7  0.341061
8     0.8  0.321425
9     0.9  0.300281
10    1.0  0.284819
Итоговое значение alpha для теста: 0.4


#   Генерация ответов для тестового набора

In [42]:
print(f"Генерация ответов для {len(test)} тестовых запросов с alpha={best_alpha:.2f}...")

test['answer'] = test['query_text'].apply(lambda q: retrieve(q, alpha=best_alpha))

test['answer'] = test['answer'].astype(str)

empty_answers = test[test['answer'].str.strip() == '']
if len(empty_answers) > 0:
    print(f"Внимание: {len(empty_answers)} запросов не дали результатов. Остается пустая строка.")

print("\nПримеры ответов (первые 3):")
for i in range(min(3, len(test))):
    print(f"query_id={test.iloc[i]['query_id']}: {test.iloc[i]['answer']}")

Генерация ответов для 500 тестовых запросов с alpha=0.40...

Примеры ответов (первые 3):
query_id=1: 2196 2511 4396 4387 2944 3843 4361 1899 3565 3168
query_id=2: 3149 4009 1923 2521 4407 3168 2944 4403 2802 3843
query_id=3: 4396 4431 4433 4408 4234 4329 4362 4045 4367 4308


# Сохранение результата в CSV

In [43]:
output_df = test[['query_id', 'answer']].copy()

output_path = "answer.csv"
output_df.to_csv(output_path, index=False, encoding='utf-8')

print(f"\nРезультат сохранён в файл {output_path}")
print(f"Количество строк: {len(output_df)}")
print("Первые 5 строк файла:")
print(output_df.head())

if set(test['query_id']) == set(output_df['query_id']):
    print("Все query_id из test присутствуют в ответе.")
else:
    missing = set(test['query_id']) - set(output_df['query_id'])
    print(f"Внимание: отсутствуют query_id: {missing}")


Результат сохранён в файл answer.csv
Количество строк: 500
Первые 5 строк файла:
   query_id                                             answer
0         1  2196 2511 4396 4387 2944 3843 4361 1899 3565 3168
1         2  3149 4009 1923 2521 4407 3168 2944 4403 2802 3843
2         3  4396 4431 4433 4408 4234 4329 4362 4045 4367 4308
3         4  4400 2831 2866 4234 2521 2865 4219 2646 3149 2696
4         5  4334 4331 4320 4234 4009 4219 1923 2232 2698 4214
Все query_id из test присутствуют в ответе.
